# Step 43 — Candidate origin association across hypocenter indexes

This step groups source **origins**, not waveforms or arrival picks. It does not associate these groups with step-32 waveform containers. Both source solutions and source line references are retained; a preferred solution is selected by source preference, not recomputed.

## Algorithm and limitations

1. Combine both indexes, parse recorded origin times into a UTC representation and sort by time. No independent clock correction is performed here.
2. Take the earliest unassigned origin as the anchor. Search later unassigned origins within `time_tol` seconds, and include those within `dist_tol` horizontal kilometres of that anchor.
3. Mark members assigned, then repeat. This is greedy anchor-based clustering, not transitive linkage or one-to-one cross-source matching. Multiple origins from the **same source** can be grouped; multiple earthquakes in a swarm could be combined if tolerances are too permissive. Membership does not require every pair to be mutually close.
4. Select the first origin from `preferred_source`, or the first member if that source is absent. Depth and magnitude are neither matching criteria nor quality-ranking criteria. Horizontal distance uses ObsPy's spherical angular-distance conversion, not a 3-D hypocentral distance.

Sequential event/origin IDs depend on input order and rows; they are not permanent external identifiers. Source-composition reports show both-source/source-only containers, not an independently validated earthquake count. Check multiorigin same-source groups and time/location differences after running.

## Inputs and safeguards

Only rows with parseable times and finite geographic coordinates should enter automatic association. This notebook now checks those prerequisites before invoking the preserved matching algorithm and stops with a message if unresolved records exist. Original reported times and unresolved annotated records remain in their source indexes for review. Explicitly curate a time-resolved input rather than silently dropping or shifting them. Depth/magnitude may remain missing; there is no automatic quality exclusion.

Outputs replace event and origin CSVs plus optional diagnostic CSVs beneath the external output root. Source-file/line references remain, but not every quality/annotation field is copied to the event table: retain source indexes alongside outputs.

## Magnitude policy (11 November 2026)

Preferred magnitude is chosen independently of the preferred origin: use a nonmissing PINAALL value first, otherwise a nonmissing summary-file value, otherwise leave magnitude missing. Legacy 0.0 values are provisionally interpreted as missing, not as true M0 earthquakes; negative nonzero magnitudes remain valid. `zero_magnitude_is_missing=False` reverses this policy for future review. Source indexes retain their raw reported values; the normalized origin table preserves them in `reported_magnitude` and records a missing-value reason. The event table records magnitude source and origin ID. Multiple candidate magnitudes within a group still require association review; first valid source value wins.

Missing numeric values are empty CSV fields (pandas NaN), not literal strings `None`, `NULL` or zero. Catalogue export omits absent magnitudes. This is a documented working interpretation of archival zeros, not proof that a physical magnitude of exactly zero is impossible. Rerun 43 and then 50 to regenerate outputs; existing catalogues are not modified by changing this notebook.


## 1. Configuration

Review source paths, selected PINAALL variant and output paths before running. Configuration does not execute the workflow.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    hypo40=P['SUMMARY_FILE_40_INDEX'],
    hypo41=P['SUMMARY_FILE_41_INDEX'],
    time_tol=float(P['TIME_TOL_S']),
    dist_tol=float(P['DIST_TOL_KM']),
    preferred_source=P['PREFERRED_SOURCE'],
    preferred_magnitude_source='pinaall',
    zero_magnitude_is_missing=True,
    out_event_csv=P['HYPO71_EVENT_INDEX'],
    out_origin_csv=P['HYPO71_ORIGIN_INDEX'],
    emit_diagnostics=True,
    diagnostics_dir=f"{P['QC_DIR']}/step43_hypo_associate_diagnostics",
)
display(SETTINGS)

Configuration: /Users/thompsong/Developer/Pinatubo1991SeismicData/10_conversion/conversion_config.json
External outputs: /Users/thompsong/work/Pinatubo1991Processing/converted


{'hypo40': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/40_summary_file_index.csv',
 'hypo41': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/41_summary_file_index.csv',
 'time_tol': 5.0,
 'dist_tol': 15.0,
 'preferred_source': 'hypo40',
 'preferred_magnitude_source': 'pinaall',
 'zero_magnitude_is_missing': True,
 'out_event_csv': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/43_hypo71_event_index.csv',
 'out_origin_csv': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/43_hypo71_origin_index.csv',
 'emit_diagnostics': True,
 'diagnostics_dir': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/qc/step43_hypo_associate_diagnostics'}

## 2. Implementation

The complete step-specific workflow is defined below. Definitions alone do not read sources or write products.

In [2]:
from __future__ import annotations

from pathlib import Path
from typing import List, Dict

import pandas as pd
from obspy.geodetics import locations2degrees, degrees2kilometers


# -----------------------------------------------------------------------------
# Utilities
# -----------------------------------------------------------------------------




# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------

def run(
    *,
    hypo40,
    hypo41,
    time_tol,
    dist_tol,
    preferred_source='hypo40',
    preferred_magnitude_source='pinaall',
    zero_magnitude_is_missing=True,
    out_event_csv,
    out_origin_csv,
    emit_diagnostics=False,
    diagnostics_dir=None
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        hypo40=hypo40,
        hypo41=hypo41,
        time_tol=time_tol,
        dist_tol=dist_tol,
        preferred_source=preferred_source,
        out_event_csv=out_event_csv,
        out_origin_csv=out_origin_csv,
        emit_diagnostics=emit_diagnostics,
        diagnostics_dir=diagnostics_dir,
    )
    if preferred_source not in ['hypo40', 'pinaall']:
        raise ValueError("Invalid preferred_source")

    # ------------------------------------------------------------------
    # Load + normalize inputs
    # ------------------------------------------------------------------

    df40 = pd.read_csv(args.hypo40)
    df41 = pd.read_csv(args.hypo41)

    df40_norm = pd.DataFrame({
        "origin_id": df40.index.map(lambda i: f"hypo40_{i}"),
        "origin_time": pd.to_datetime(df40["origin_time"], format="mixed", utc=True),
        "latitude": df40["latitude"].astype(float),
        "longitude": df40["longitude"].astype(float),
        "depth_km": df40["depth_km"].astype(float),
        "magnitude": df40["magnitude"].astype(float),
        "source": "hypo40",
        "source_file": df40["source_file"],
        "source_line": df40["source_line"],
    })

    df41_norm = pd.DataFrame({
        "origin_id": df41.index.map(lambda i: f"pinaall_{i}"),
        "origin_time": pd.to_datetime(df41["origin_time"], format="mixed", utc=True),
        "latitude": df41["latitude"].astype(float),
        "longitude": df41["longitude"].astype(float),
        "depth_km": df41["depth_km"].astype(float),
        "magnitude": df41["magnitude"].astype(float),
        "source": "pinaall",
        "source_file": df41["source_file"],
        "source_line": df41["source_line"],
    })

    for label, frame in [('hypo40', df40_norm), ('pinaall', df41_norm)]:
        import numpy as np
        valid = (frame['origin_time'].notna() & np.isfinite(frame['latitude'])
                 & np.isfinite(frame['longitude']) & frame['latitude'].between(-90, 90)
                 & frame['longitude'].between(-180, 180))
        if not valid.all():
            raise ValueError(f'{label}: {int((~valid).sum())} unresolved time/coordinate rows; review and explicitly select valid input before association.')
    if preferred_magnitude_source not in ('hypo40', 'pinaall'):
        raise ValueError('Unknown preferred magnitude source')
    for frame in (df40_norm, df41_norm):
        frame['reported_magnitude'] = frame['magnitude']
        frame['magnitude_missing_reason'] = ''
        zero = frame['magnitude'].eq(0) & zero_magnitude_is_missing
        frame.loc[zero, 'magnitude'] = float('nan')
        frame.loc[zero, 'magnitude_missing_reason'] = 'reported_zero_treated_as_missing'
    all_origins = (
        pd.concat([df40_norm, df41_norm], ignore_index=True)
        .sort_values("origin_time")
        .reset_index(drop=True)
    )

    # ------------------------------------------------------------------
    # Event association (UNCHANGED)
    # ------------------------------------------------------------------

    events: List[Dict] = []
    origins_out: List[Dict] = []

    event_id = 0
    assigned = [False] * len(all_origins)

    for i, row in all_origins.iterrows():
        if assigned[i]:
            continue

        event_id += 1
        assigned[i] = True
        members = [i]

        for j in range(i + 1, len(all_origins)):
            if assigned[j]:
                continue

            dt = abs(
                (all_origins.loc[j, "origin_time"] - row["origin_time"])
                .total_seconds()
            )
            if dt > args.time_tol:
                break

            dist = degrees2kilometers(locations2degrees(
                row["latitude"], row["longitude"],
                all_origins.loc[j, "latitude"],
                all_origins.loc[j, "longitude"]
            ))

            if dist <= args.dist_tol:
                assigned[j] = True
                members.append(j)

        group = all_origins.loc[members]

        preferred = group[group["source"] == args.preferred_source]
        preferred = preferred.iloc[[0]] if not preferred.empty else group.iloc[[0]]

        valid_magnitudes = group[group['magnitude'].notna()]
        preferred_magnitudes = valid_magnitudes[valid_magnitudes['source'].eq(preferred_magnitude_source)]
        magnitude_row = (preferred_magnitudes.iloc[0] if not preferred_magnitudes.empty
                         else valid_magnitudes.iloc[0] if not valid_magnitudes.empty else None)
        events.append({
            "event_id": event_id,
            "preferred_origin_time": preferred.iloc[0]["origin_time"].isoformat(),
            "preferred_latitude": preferred.iloc[0]["latitude"],
            "preferred_longitude": preferred.iloc[0]["longitude"],
            "preferred_depth_km": preferred.iloc[0]["depth_km"],
            "preferred_magnitude": magnitude_row['magnitude'] if magnitude_row is not None else float('nan'),
            "preferred_magnitude_source": magnitude_row['source'] if magnitude_row is not None else None,
            "preferred_magnitude_origin_id": magnitude_row['origin_id'] if magnitude_row is not None else None,
            "magnitude_selection_policy": 'pinaall_first_nonzero_then_other_source' if zero_magnitude_is_missing else 'preferred_source_first',
            "zero_magnitude_is_missing": bool(zero_magnitude_is_missing),
            "n_origins": len(group),
        })

        for _, o in group.iterrows():
            origins_out.append({
                "event_id": event_id,
                "origin_id": o["origin_id"],
                "origin_time": o["origin_time"].isoformat(),
                "latitude": o["latitude"],
                "longitude": o["longitude"],
                "depth_km": o["depth_km"],
                "magnitude": o["magnitude"],
                "reported_magnitude": o['reported_magnitude'],
                "magnitude_missing_reason": o['magnitude_missing_reason'],
                "source": o["source"],
                "source_file": o["source_file"],
                "source_line": o["source_line"],
            })

    # ------------------------------------------------------------------
    # Write primary outputs
    # ------------------------------------------------------------------

    out_event = Path(args.out_event_csv)
    out_origin = Path(args.out_origin_csv)
    out_event.parent.mkdir(parents=True, exist_ok=True)

    df_events = pd.DataFrame(events)
    df_origins = pd.DataFrame(origins_out)

    out_origin.parent.mkdir(parents=True, exist_ok=True)
    df_events.to_csv(out_event, index=False)
    df_origins.to_csv(out_origin, index=False)

    # ------------------------------------------------------------------
    # Diagnostics (NEW)
    # ------------------------------------------------------------------

    comp = (
        df_origins
        .groupby(["event_id", "source"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    comp["has_hypo40"] = comp.get("hypo40", 0) > 0
    comp["has_pinaall"] = comp.get("pinaall", 0) > 0

    both = comp[comp["has_hypo40"] & comp["has_pinaall"]]
    hypo40_only = comp[comp["has_hypo40"] & ~comp["has_pinaall"]]
    pinaall_only = comp[~comp["has_hypo40"] & comp["has_pinaall"]]

    # ------------------------------------------------------------------
    # Report
    # ------------------------------------------------------------------

    print("\nSTEP 43 — HYPOCENTER ASSOCIATION COMPLETE")
    print("----------------------------------------")
    print(f"Total events:            {len(df_events)}")
    print(f"Total origins:           {len(df_origins)}")
    print("")
    print("FUZZY RECONCILIATION SUMMARY")
    print("============================")
    print(f"Events with both sources: {len(both)}")
    print(f"Events with hypo40 only:  {len(hypo40_only)}")
    print(f"Events with pinaall only: {len(pinaall_only)}")
    print("")
    print("ORIGIN COUNTS")
    print("-------------")
    print(f"Hypo40 origins:  {(df_origins['source'] == 'hypo40').sum()}")
    print(f"Pinaall origins: {(df_origins['source'] == 'pinaall').sum()}")
    print("")
    print(f"Event CSV:  {out_event}")
    print(f"Origin CSV: {out_origin}")

    # ------------------------------------------------------------------
    # Optional diagnostic CSVs
    # ------------------------------------------------------------------

    if args.emit_diagnostics and args.diagnostics_dir:
        diag_dir = Path(args.diagnostics_dir)
        diag_dir.mkdir(parents=True, exist_ok=True)

        comp.to_csv(diag_dir / "43_event_source_composition.csv", index=False)
        both.to_csv(diag_dir / "43_events_with_both_sources.csv", index=False)
        hypo40_only.to_csv(diag_dir / "43_events_hypo40_only.csv", index=False)
        pinaall_only.to_csv(diag_dir / "43_events_pinaall_only.csv", index=False)

        print("\nDiagnostic CSVs written to:")
        print(f"  {diag_dir}")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## 3. Execute when ready

This cell reads configured inputs and replaces configured outputs. Inspect summaries and diagnostic tables after the run.

In [3]:
result = run(**SETTINGS)


STEP 43 — HYPOCENTER ASSOCIATION COMPLETE
----------------------------------------
Total events:            4690
Total origins:           8233

FUZZY RECONCILIATION SUMMARY
Events with both sources: 3500
Events with hypo40 only:  1105
Events with pinaall only: 85

ORIGIN COUNTS
-------------
Hypo40 origins:  4632
Pinaall origins: 3601

Event CSV:  /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/43_hypo71_event_index.csv
Origin CSV: /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/43_hypo71_origin_index.csv

Diagnostic CSVs written to:
  /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/qc/step43_hypo_associate_diagnostics


In [4]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)


df40


,origin_time,latitude,longitude,depth_km,magnitude,nass,rms,source_file,source_line,parser,raw_line
0,1991-05-07T03:21:14.290000,15.168667,120.322333,3.28,1.36,7,0.10,Pinatubo_all.sum,1,vdap_hypo71_adapter_v2,91 5 7 321 14.29 15n10.12 120E19.34 3.28 ...
1,1991-05-07T12:52:29.950000,15.169833,120.313500,1.89,0.70,7,0.10,Pinatubo_all.sum,2,vdap_hypo71_adapter_v2,91 5 7 1252 29.95 15n10.19 120E18.81 1.89 ...
2,1991-05-08T02:37:00.060000,15.163333,120.331167,4.55,1.00,7,0.08,Pinatubo_all.sum,3,vdap_hypo71_adapter_v2,91 5 8 237 0.06 15n 9.80 120E19.87 4.55 ...
3,1991-05-08T17:56:33.040000,15.158000,120.317833,4.53,1.04,7,0.07,Pinatubo_all.sum,4,vdap_hypo71_adapter_v2,91 5 8 1756 33.04 15n 9.48 120E19.07 4.53 ...
4,1991-05-08T17:59:23.980000,15.154667,120.330167,4.07,0.49,8,0.11,Pinatubo_all.sum,5,vdap_hypo71_adapter_v2,91 5 8 1759 23.98 15n 9.28 120E19.81 4.07 ...


df41


,origin_time,latitude,longitude,depth_km,magnitude,source_file,source_line,parser,raw_line
0,1991-05-10T04:10:52.990000,15.440667,120.380500,15.00,0.00,PINAALL.DAT,1,pinaall_dat_v3,91 510 410 52.99 15n26.44 120e22.83 15.00 ...
1,1991-05-10T05:15:26.520000,15.155667,120.326000,3.37,0.93,PINAALL.DAT,2,pinaall_dat_v3,91 510 515 26.52 15n 9.34 120e19.56 3.37 ...
2,1991-05-11T21:18:53.740000,15.152833,120.321833,2.53,0.88,PINAALL.DAT,3,pinaall_dat_v3,91 511 2118 53.74 15n 9.17 120e19.31 2.53 ...
3,1991-05-12T02:04:25.200000,15.166833,120.328167,3.29,0.79,PINAALL.DAT,4,pinaall_dat_v3,91 512 2 4 25.20 15n10.01 120e19.69 3.29 ...
4,1991-05-16T18:52:22.920000,15.134667,120.339833,1.35,0.68,PINAALL.DAT,5,pinaall_dat_v3,91 516 1852 22.92 15n 8.08 120e20.39 1.35 ...


df40_norm


,origin_id,origin_time,latitude,longitude,depth_km,magnitude,source,source_file,source_line,reported_magnitude,magnitude_missing_reason
0,hypo40_0,1991-05-07 03:21:14.290000+00:00,15.168667,120.322333,3.28,1.36,hypo40,Pinatubo_all.sum,1,1.36,
1,hypo40_1,1991-05-07 12:52:29.950000+00:00,15.169833,120.313500,1.89,0.70,hypo40,Pinatubo_all.sum,2,0.70,
2,hypo40_2,1991-05-08 02:37:00.060000+00:00,15.163333,120.331167,4.55,1.00,hypo40,Pinatubo_all.sum,3,1.00,
3,hypo40_3,1991-05-08 17:56:33.040000+00:00,15.158000,120.317833,4.53,1.04,hypo40,Pinatubo_all.sum,4,1.04,
4,hypo40_4,1991-05-08 17:59:23.980000+00:00,15.154667,120.330167,4.07,0.49,hypo40,Pinatubo_all.sum,5,0.49,


... (4 outputs left out)

df_events


,event_id,preferred_origin_time,preferred_latitude,preferred_longitude,preferred_depth_km,preferred_magnitude,preferred_magnitude_source,preferred_magnitude_origin_id,magnitude_selection_policy,zero_magnitude_is_missing,n_origins
0,1,1991-05-07T03:21:14.290000+00:00,15.168667,120.322333,3.28,1.36,hypo40,hypo40_0,pinaall_first_nonzero_then_other_source,True,1
1,2,1991-05-07T12:52:29.950000+00:00,15.169833,120.313500,1.89,0.70,hypo40,hypo40_1,pinaall_first_nonzero_then_other_source,True,1
2,3,1991-05-08T02:37:00.060000+00:00,15.163333,120.331167,4.55,1.00,hypo40,hypo40_2,pinaall_first_nonzero_then_other_source,True,1
3,4,1991-05-08T17:56:33.040000+00:00,15.158000,120.317833,4.53,1.04,hypo40,hypo40_3,pinaall_first_nonzero_then_other_source,True,1
4,5,1991-05-08T17:59:23.980000+00:00,15.154667,120.330167,4.07,0.49,hypo40,hypo40_4,pinaall_first_nonzero_then_other_source,True,1


df_origins


,event_id,origin_id,origin_time,latitude,longitude,depth_km,magnitude,reported_magnitude,magnitude_missing_reason,source,source_file,source_line
0,1,hypo40_0,1991-05-07T03:21:14.290000+00:00,15.168667,120.322333,3.28,1.36,1.36,,hypo40,Pinatubo_all.sum,1
1,2,hypo40_1,1991-05-07T12:52:29.950000+00:00,15.169833,120.313500,1.89,0.70,0.70,,hypo40,Pinatubo_all.sum,2
2,3,hypo40_2,1991-05-08T02:37:00.060000+00:00,15.163333,120.331167,4.55,1.00,1.00,,hypo40,Pinatubo_all.sum,3
3,4,hypo40_3,1991-05-08T17:56:33.040000+00:00,15.158000,120.317833,4.53,1.04,1.04,,hypo40,Pinatubo_all.sum,4
4,5,hypo40_4,1991-05-08T17:59:23.980000+00:00,15.154667,120.330167,4.07,0.49,0.49,,hypo40,Pinatubo_all.sum,5
